In [1]:
import pandas as pd 
import numpy as np
import glob
from pathlib import Path 

In [2]:
df = pd.read_csv("data/rmg_employee_dirty.csv")

print(df.shape)
print(df.dtypes)
print(df.isna().sum())
print(df.duplicated().sum())

(5120, 17)
employee_id        object
employee_name      object
age                object
gender             object
blood_group        object
department         object
designation        object
division           object
basic_salary       object
bonus               int64
total_salary      float64
phone              object
join_date          object
attendance_pct    float64
email              object
nid               float64
experience_yrs    float64
dtype: object
employee_id         0
employee_name       0
age               101
gender             37
blood_group        72
department          0
designation        76
division            0
basic_salary       16
bonus               0
total_salary      101
phone             111
join_date          57
attendance_pct    154
email              36
nid                81
experience_yrs     39
dtype: int64
111


## All Diagnosis Functions

In [3]:
def column_based_inspection(df, column, *, show_nulls=False, show_duplicates=False):
    """
    Prints a quick diagnostic summary of a specific DataFrame column,
    with optional inline data sampling for debugging.
    """
    if column not in df.columns:
        print(f"❌ Error: Column '{column}' does not exist in this DataFrame.")
        return

    print(f"\n📊 --- Inspection Report: {column} ---")
    print(f"Total Rows:     {df[column].shape[0]}")
    print(f"Data Type:      {df[column].dtype}")
    print(f"Null Values:    {df[column].isnull().sum()}")
    print(f"Duplicate Rows: {df[column].duplicated().sum()}\n")

    # Reuse 'column' automatically if the user toggles the flag to True
    if show_nulls and df[column].isnull().sum() > 0:
        print(f"🔍 --- Null Data Sample (First 10 Rows) ---")
        print(f"{df[df[column].isna()].head(10)}\n")
        
    if show_duplicates and df[column].duplicated().sum() > 0:
        print(f"👯 --- Duplicate Data Sample (First 10 Rows) ---")
        # Added keep=False so you can see BOTH copies of the duplicates side-by-side
        print(f"{df[df[column].duplicated(keep=False)].sort_values(by=column).head(10)}\n")

In [4]:
column_based_inspection(df, 'employee_id', show_nulls=True, show_duplicates=True)


📊 --- Inspection Report: employee_id ---
Total Rows:     5120
Data Type:      object
Null Values:    0
Duplicate Rows: 111

👯 --- Duplicate Data Sample (First 10 Rows) ---
     employee_id      employee_name age  gender blood_group department  \
30      EMP00031       Rashid Akter  48    Male          O+  Finishing   
5026    EMP00031       Rashid Akter  48    Male          O+  Finishing   
85      EMP00086        Sohel Begum  20    Male         AB-      Admin   
5103    EMP00086        Sohel Begum  20    Male         AB-      Admin   
5058    EMP00137  ABDUL KARIM BEGUM  32    Male          A-     Sweing   
136     EMP00137  ABDUL KARIM BEGUM  32    Male          A-     Sweing   
218     EMP00219        Monira Khan  21    Male          B-    cutting   
5104    EMP00219        Monira Khan  21    Male          B-    cutting   
5102    EMP00220      Sharmin Uddin  21  Female          A-     Sewing   
219     EMP00220      Sharmin Uddin  21  Female          A-     Sewing   

            

In [ ]:

def check_global_structure(df):
    """
    Global check for completely identical row duplicates and constant columns.
    Run this first before diagnosing individual columns.
    """
    print("\n=== Global Dataset Structure Audit ===")
    print(f"Total Rows:    {len(df)}")
    print(f"Total Columns: {len(df.columns)}")
    
    row_dupes = df.duplicated().sum()
    print(f"Completely Duplicated Rows: {row_dupes} ({(row_dupes/len(df))*100:.2f}%)")
    
    print("\nChecking for constant (single-value) columns...")
    constant_cols = []
    for col in df.columns:
        if df[col].nunique() == 1:
            constant_cols.append(col)
            val = df[col].dropna().unique()[0]
            print(f"  - '{col}' is constant. Every row contains: '{val}'")
            
    if not constant_cols:
        print("  No constant columns found. All features contain variation.")


def find_all_hidden_nulls(df):
    """
    Scans all text/categorical columns for pseudo-null string placeholders.
    Bypasses pandas' native .isna() to find strings like '?', 'none', 'n/a'.
    """
    print("\n=== Global Hidden Null Scan ===")
    
    # Only scan object/string columns; numeric columns can't hold these strings anyway
    text_cols = df.select_dtypes(include=['object', 'category', 'string']).columns
    null_variants = ['?', 'none', 'n/a', 'null', 'nan', 'nil', 'not available', '-']
    
    found_any = False
    
    for col in text_cols:
        col_clean = df[col].astype(str).str.strip().str.lower()
        
        # Mask: Matches a placeholder AND isn't already a true pandas NaN
        mask = col_clean.isin(null_variants) & df[col].notna()
        hidden_count = mask.sum()
        
        if hidden_count > 0:
            found_any = True
            print(f"\n'{col}':")
            print(f"  pandas native nulls: {df[col].isna().sum()}")
            print(f"  hidden string nulls: {hidden_count}")
            print(f"  breakdown:")
            # Use to_string() to keep the output flat and clean
            print(df[col][mask].value_counts().to_string())
            
    if not found_any:
        print("\n  No hidden string placeholders detected in any text columns.")



In [6]:
def diagnose_identifier(df, column, expected_lengths=None, check_duplicates=True):
    """
    Universal identifier column diagnosis.
    Works on phone, NID, employee ID, customer ID, order ID — anything.
    
    expected_lengths: list of valid lengths e.g. [11] for BD phone, [10,13] for NID
    """
    print(f"\n'{column}':")
    print(f"  dtype : {df[column].dtype}")
    print(f"  nulls : {df[column].isna().sum()}")
    
    # Normalize to string for analysis
    col_str = df[column].dropna().astype(str).str.strip()
    # Remove scientific notation artifacts (e.g. from NID read as float)
    col_str = col_str.str.replace(r'\.0$', '', regex=True)
    
    # Length distribution
    print(f"\n  Value length distribution:")
    print(col_str.str.len().value_counts().sort_index().to_string())
    
    # Flag unexpected lengths
    if expected_lengths:
        unexpected = col_str[~col_str.str.len().isin(expected_lengths)]
        print(f"\n  Values with unexpected length: {len(unexpected)}")
        if len(unexpected) > 0:
            print(f"  Sample: {unexpected.head(5).tolist()}")
    
    # Duplicate check
    if check_duplicates:
        dupes = col_str[col_str.duplicated(keep=False)]
        print(f"\n  Duplicate values: {len(dupes)}")
        if len(dupes) > 0:
            print(f"  Sample duplicates: {dupes.head(10).tolist()}")
    
    # Sample raw values
    print(f"\n  Sample raw values:")
    print(df[column].dropna().head(10).tolist())



In [7]:

def find_non_numeric(df,column):
    converted = pd.to_numeric(df[column], errors='coerce')
    mask = df[column].notna() & converted.isna()
    bad_values = df[column][mask].value_counts()
    print(bad_values)



def diagnose_dates(df, column):
    print(f"\n'{column}':")
    print(f"  total rows  : {len(df)}")
    print(f"  nulls       : {df[column].isna().sum()}")
    
    # Try parsing — anything that fails becomes NaT
    parsed = pd.to_datetime(df[column], format='mixed', errors='coerce')
    unparseable = df[column].notna() & parsed.isna()
    
    print(f"  unparseable : {unparseable.sum()}")
    print(f"  successfully parsed: {parsed.notna().sum()}")
    
    print(f"\n  Unparseable values:")
    print(df[column][unparseable].value_counts())
    
    print(f"\n  Sample of parsed dates (min/max):")
    print(f"  earliest: {parsed.min()}")
    print(f"  latest  : {parsed.max()}")


In [8]:
def check_numeric_ranges(df, column, min_value=None, max_value=None):
    numeric_col = pd.to_numeric(df[column], errors='coerce')
    print(f"\n'{column}' — distribution:")
    print(f"  min:    {numeric_col.min()}")
    print(f"  max:    {numeric_col.max()}")
    print(f"  mean:   {numeric_col.mean():.2f}")
    print(f"  median: {numeric_col.median()}")
    print(f"  nulls:  {numeric_col.isna().sum()}")

    if min_value is not None:
        below = (numeric_col < min_value).sum()
        print(f"  rows below {min_value}: {below}")
    if max_value is not None:
        above = (numeric_col > max_value).sum()
        print(f"  rows above {max_value}: {above}")



# Statistical outlier detection — values beyond 3 standard deviations
def find_statistical_outliers(df, column):
    numeric_col = pd.to_numeric(df[column], errors='coerce').dropna()
    mean = numeric_col.mean()
    std = numeric_col.std()
    
    lower = mean - 3 * std
    upper = mean + 3 * std
    
    outliers = numeric_col[(numeric_col < lower) | (numeric_col > upper)]
    print(f"\n'{column}' — statistical outliers (beyond 3 std):")
    print(f"  expected range: {lower:.2f} to {upper:.2f}")
    print(f"  outlier count:  {len(outliers)}")
    print(f"  outlier values: {outliers.values}")


In [9]:
def find_category_inconsistencies(df, column, top_n=20):
    print(f"\n'{column}' — top {top_n} values (raw):")
    print(df[column].value_counts().head(top_n))

        # Now normalize and compare
    normalized = df[column].astype(str).str.strip().str.lower()
    print(f"\n'{column}' — top {top_n} values (normalized):")
    print(normalized.value_counts().head(top_n))

       # The difference between these two outputs = your inconsistency problem
    raw_unique = df[column].nunique()
    normalized_unique = normalized.nunique()
    print(f"\n  unique values raw:        {raw_unique}")
    print(f"  unique values normalized: {normalized_unique}")
    print(f"  collapsible duplicates:   {raw_unique - normalized_unique}")



In [10]:
import pandas as pd

def find_mixed_script_names(df, name_column):
    """
    Filters and returns rows where the name contains both 
    Bengali characters and English characters.
    """
    # Defensive copy
    df = df.copy()
    
    # 1. Define Regex patterns for both languages
    # Unicode block for Bengali: \u0980 to \u09FF
    bengali_pattern = r'[\u0980-\u09FF]'
    english_pattern = r'[a-zA-Z]'
    
    # 2. Check if a row contains Bengali AND contains English
    contains_bengali = df[name_column].astype(str).str.contains(bengali_pattern, regex=True)
    contains_english = df[name_column].astype(str).str.contains(english_pattern, regex=True)
    
    # 3. Create a mask where both conditions are True
    mixed_mask = contains_bengali & contains_english
    
    # Filter the dataframe to isolate these rows
    mixed_script_df = df[mixed_mask]
    
    print(f"🔍 Found {len(mixed_script_df)} names with mixed Bengali and English scripts.")
    return mixed_script_df

In [11]:

def check_salary_discrepancy(df, basic_col, bonus_col, total_col):
    """
    Checks if basic + bonus equals total salary.
    Returns a DataFrame containing only the rows where the math doesn't add up.
    """
    # 1. Safely fill NaNs with 0 purely for the math calculation
    # (This does NOT alter your actual dataframe)
    expected_total = df[basic_col].fillna(0) + df[bonus_col].fillna(0)
    actual_total = df[total_col].fillna(0)
    
    # 2. Round to 2 decimals to prevent floating point errors (e.g., 10.00001 != 10.0)
    expected_total = expected_total.round(2)
    actual_total = actual_total.round(2)
    
    # 3. Create a mask where the calculated total does NOT equal the given total
    discrepancy_mask = expected_total != actual_total
    
    # 4. Count the errors
    error_count = discrepancy_mask.sum()
    
    print(f"📊 --- Discrepancy Report ---")
    print(f"Equation Checked: {basic_col} + {bonus_col} = {total_col}")
    print(f"Total Rows with Math Errors: {error_count}\n")
    
    # 5. Return only the corrupted rows for the user to investigate
    if error_count > 0:
        return df[discrepancy_mask][[basic_col, bonus_col, total_col]]
    else:
        print("✅ All rows match perfectly!")
        return None

In [12]:
check_global_structure(df)
find_all_hidden_nulls(df)


=== Global Dataset Structure Audit ===
Total Rows:    5120
Total Columns: 17
Completely Duplicated Rows: 111 (2.17%)

Checking for constant (single-value) columns...
  No constant columns found. All features contain variation.

=== Global Hidden Null Scan ===

  No hidden string placeholders detected in any text columns.


In [13]:
diagnose_identifier(df, 'employee_id')


'employee_id':
  dtype : object
  nulls : 0

  Value length distribution:
employee_id
6     200
8    4920

  Duplicate values: 222
  Sample duplicates: ['EMP00031', 'EMP00086', 'EMP00137', 'EMP00219', 'EMP00220', 'EMP00244', 'EMP00313', 'EMP00321', 'EMP00351', 'EMP00371']

  Sample raw values:
['EMP00001', 'EMP00002', 'EMP00003', 'EMP00004', 'EMP00005', 'E-0006', 'EMP00007', 'EMP00008', 'EMP00009', 'EMP00010']


In [14]:

def remove_duplicates(df, subset = None):
         df = df.copy()
         before = len(df)
         df = df.drop_duplicates(subset=subset, ignore_index=True)
         after = len(df)
         print(f"Duplicates handling ==> Total removed rows: {before - after} | Remaining rows: {after}")
         return df 
    


In [15]:
df_cleaned = remove_duplicates(df)


Duplicates handling ==> Total removed rows: 111 | Remaining rows: 5009


In [16]:
def cleaning_columns(df):
         df = df.copy()
         df.columns = (df.columns.str.strip().str.lower().str.replace(' ', '_', regex = False))
         print("Cleaned column names")
         return df

In [17]:
df_cleaned = cleaning_columns(df_cleaned)




Cleaned column names


In [18]:
diagnose_identifier(df_cleaned, 'employee_id')


'employee_id':
  dtype : object
  nulls : 0

  Value length distribution:
employee_id
6     200
8    4809

  Duplicate values: 0

  Sample raw values:
['EMP00001', 'EMP00002', 'EMP00003', 'EMP00004', 'EMP00005', 'E-0006', 'EMP00007', 'EMP00008', 'EMP00009', 'EMP00010']


In [19]:
# # employee id cleaning code


# df_cleaned['employee_id']= df_cleaned['employee_id'].astype(str).str.replace('E-', 'EMP0')

# df_cleaned = df_cleaned.drop_duplicates().reset_index(drop=True)

# df_cleaned.loc[(df_cleaned['employee_id'] ==  'EMP01147') & (df_cleaned['experience_yrs'] == 99.0), 'experience_yrs'] = np.nan

# # Sort so the real value (10.0) comes first, NaN goes last
# df_cleaned = df_cleaned.sort_values('experience_yrs')

# # Now drop duplicates keeping first — first row has 10.0, NaN gets dropped
# df_cleaned = df_cleaned.drop_duplicates(
#     subset=[col for col in df_cleaned.columns if col != 'experience_yrs'],
#     keep='first'
# ).reset_index(drop=True)

In [ ]:
def clean_employee_id_bulletproof(df, target_col='employee_id'):
    """
    Standardizes and cleans identification columns to a unified format.

    This function isolates non-null entries to prevent string corruption,
    strips whitespace, forces uppercase, standardizes hyphen characters, 
    and uses regex anchoring to safely replace leading prefixes (e.g., 'E-' to 'EMP0').
    """

    df = df.copy()
    
    if target_col not in df.columns:
        print(f"❌ Error: '{target_col}' not found. Check your column names.")
        return df
   
    is_not_null = df[target_col].notna()
    
    cleaned_series = (df.loc[is_not_null, target_col]
                      .astype(str)
                      .str.strip()
                      .str.upper()
                      .str.replace('–', '-', regex=False))
  
    cleaned_series = cleaned_series.str.replace(r'^E-', 'EMP0', regex=True)
    df.loc[is_not_null, target_col] = cleaned_series
    
    print(f"✅ Column '{target_col}' successfully cleaned and anchored.")
    return df

In [21]:
df_cleaned = clean_employee_id_bulletproof(df_cleaned)
diagnose_identifier(df_cleaned, 'employee_id')

✅ Column 'employee_id' successfully cleaned and anchored.

'employee_id':
  dtype : object
  nulls : 0

  Value length distribution:
employee_id
8    5009

  Duplicate values: 18
  Sample duplicates: ['EMP00386', 'EMP00522', 'EMP01147', 'EMP02021', 'EMP02213', 'EMP02276', 'EMP03124', 'EMP03650', 'EMP03984', 'EMP03984']

  Sample raw values:
['EMP00001', 'EMP00002', 'EMP00003', 'EMP00004', 'EMP00005', 'EMP00006', 'EMP00007', 'EMP00008', 'EMP00009', 'EMP00010']


In [22]:
column_based_inspection(df_cleaned, 'employee_id')


📊 --- Inspection Report: employee_id ---
Total Rows:     5009
Data Type:      object
Null Values:    0
Duplicate Rows: 9



In [23]:
diagnose_identifier(df_cleaned, 'phone', expected_lengths=[11])


'phone':
  dtype : object
  nulls : 110

  Value length distribution:
phone
9        9
10      11
11    4179
12     310
14     390

  Values with unexpected length: 720
  Sample: ['01782-374753', '+8801796637649', '01769-302158', '+8801964543049', '+8801773709724']

  Duplicate values: 36
  Sample duplicates: ['0000000000', '01996647103', '0000000000', '0000000000', '01986582582', '0000000000', 'not given', '0000000000', '01990-654516', '0000000000']

  Sample raw values:
['01997226012', '01820576383', '01942857966', '01963606628', '01982070937', '01812614124', '01897225156', '01782-374753', '01735529407', '01762401521']


In [24]:
def clean_phone(df, column):
    df = df.copy()
    phone = (
        df[column]
        .astype(str)
        .str.replace(r'\D', '', regex=True)
        .str.replace(r'^88', '', regex=True)
    )

    phone = phone.where((phone.str.len() == 11) & (phone.str.startswith('01')), np.nan)

    df[column] = phone

    return df

In [25]:
df_cleaned = clean_phone(df_cleaned, 'phone')


In [26]:
diagnose_identifier(df_cleaned, 'nid')


'nid':
  dtype : float64
  nulls : 80

  Value length distribution:
nid
10     192
13    4737

  Duplicate values: 99
  Sample duplicates: ['6001807519611', '3003022489752', '1289873073602', '1114246568635', '5708757547909', '6576899183883', '8383753341460', '5385682298741', '3925946215955', '2862576726462']

  Sample raw values:
[8899391642592.0, 7364547568321.0, 5028919575309.0, 9682317303976.0, 4142892825.0, 5210388713215.0, 3127947929743.0, 9317531495235.0, 4227332407137.0, 8440108492786.0]


In [ ]:


def clean_national_id(df: pd.DataFrame, col_name: str = 'nid', valid_lengths: tuple = (10, 13)) -> pd.DataFrame:
    """
    Cleans an NID column by validating lengths and converting non-meaningful 
    duplicate IDs into pd.NA while preserving the unique row data.
    
    Expected Inputs:
    - df: The target DataFrame.
    - col_name: The NID column (defaults to 'nid').
    - valid_lengths: Acceptable NID lengths (defaults to 10 or 13).
    
    Output:
    - A DataFrame where invalid lengths and duplicate NIDs are set to pd.NA.
    """
    df = df.copy()
    
    if col_name not in df.columns:
        raise KeyError(f"Column '{col_name}' not found.")
        
    temp_col = df[col_name].astype(str)
    temp_col = temp_col.str.replace(r'\.0$', '', regex=True).str.strip()
    
    # 1. Identify valid lengths
    # Upgraded validation: Must be correct length AND contain only numbers
    valid_mask = temp_col.str.len().isin(valid_lengths) & temp_col.str.isdigit()
    
    # 2. Identify duplicates (keep=False marks ALL occurrences of a duplicate as True)
    # We ignore 'nan' strings so actual nulls don't trigger the duplicate detector
    duplicate_mask = temp_col.duplicated(keep=False) & (temp_col != 'nan')
    
    # 3. Combine logic: Must be a valid length AND NOT a duplicate
    final_mask = valid_mask & (~duplicate_mask) & (temp_col != 'nan')
    
    df[col_name] = temp_col.where(final_mask, pd.NA)
    
    return df

In [28]:
df_cleaned = clean_national_id(df_cleaned)
diagnose_identifier(df_cleaned, 'nid')


'nid':
  dtype : object
  nulls : 179

  Value length distribution:
nid
10     192
13    4638

  Duplicate values: 0

  Sample raw values:
['8899391642592', '7364547568321', '5028919575309', '9682317303976', '4142892825', '5210388713215', '3127947929743', '9317531495235', '4227332407137', '8440108492786']


In [29]:
age_map = {
    'eighteen': 18,
    'nineteen': 19,
    'twenty two': 22,
    'twenty five': 25,
    'twenty eight': 28,
    'thirty': 30,
    'thirty five': 35,
    'forty': 40,
    'forty five': 45
}

In [ ]:
import pandas as pd


def fix_numeric_columns(df, columns, mapping_dict=None):
    """
    Cleans numeric columns by optionally mapping known text values to numeric
    equivalents and converting values to a numeric dtype. Invalid values are
    coerced to NaN.

    Args:
        df (pd.DataFrame): Input DataFrame.
        columns (list[str]): Columns to convert.
        mapping_dict (dict, optional): Dictionary mapping text values to
            numeric equivalents (e.g., {"thirty": 30}).

    Returns:
        pd.DataFrame: A copy of the DataFrame with cleaned numeric columns.

    Raises:
        KeyError: If a specified column does not exist.

    Example:
        >>> age_map = {"thirty": 30, "forty five": 45}
        >>> df_clean = fix_numeric_columns(
        ...     df,
        ...     columns=["age", "experience"],
        ...     mapping_dict=age_map
        ... )
    """
    df = df.copy()

    for col in columns:
        if col not in df.columns:
            raise KeyError(f"Column '{col}' not found in DataFrame")

        if mapping_dict is not None:
            df[col] = df[col].replace(mapping_dict)

        df[col] = pd.to_numeric(df[col], errors="coerce")

    print(f"Cleaned numeric columns: {columns}")
    return df

In [31]:
df_cleaned = fix_numeric_columns(df_cleaned, ['age'], mapping_dict=age_map)
column_based_inspection(df_cleaned, 'age')

Cleaned numeric columns: ['age']

📊 --- Inspection Report: age ---
Total Rows:     5009
Data Type:      float64
Null Values:    100
Duplicate Rows: 4966



In [32]:


def clean_currency_columns(df, columns):
    """
    Cleans currency columns by removing currency symbols (like ৳), words (like taka), 
    and commas, then converts the cleaned values into a proper numeric data type.
    
    Expected Inputs:
    - df: The target DataFrame.
    - columns: A list of currency column names to clean (e.g., ['basic_salary']).
    
    Output:
    - A DataFrame with the specified columns converted to clean float/numeric types.
    """
    df = df.copy()
    
    for col in columns:
        if col not in df.columns:
            raise KeyError(f"Column '{col}' not found in DataFrame.")
        
        cleaned_series = df[col].astype(str).str.replace(r'[^0-9.]', '', regex=True)
        
        df[col] = pd.to_numeric(cleaned_series, errors='coerce')
        
    return df


In [33]:
df_cleaned = clean_currency_columns(df_cleaned, ['basic_salary'])
column_based_inspection(df_cleaned, 'basic_salary')


📊 --- Inspection Report: basic_salary ---
Total Rows:     5009
Data Type:      float64
Null Values:    15
Duplicate Rows: 537



In [34]:
import pandas as pd

def fix_date_columns(df, columns):
    """
    Safely converts specified columns to datetime objects, 
    handling mixed formats and missing values gracefully.
    """
    df = df.copy()
    
    for col in columns:
        if col not in df.columns:
            raise KeyError(f"Column '{col}' not found in DataFrame.")
            
        df[col] = pd.to_datetime(df[col], format='mixed', errors='coerce')
        
    print(f"Cleaned date columns: {columns}")
    return df

In [35]:
df_cleaned = fix_date_columns(df_cleaned, ['join_date'])
column_based_inspection(df_cleaned, 'join_date', show_nulls=True, show_duplicates=True)

Cleaned date columns: ['join_date']

📊 --- Inspection Report: join_date ---
Total Rows:     5009
Data Type:      datetime64[ns]
Null Values:    109
Duplicate Rows: 3308

🔍 --- Null Data Sample (First 10 Rows) ---
    employee_id   employee_name   age  gender blood_group       department  \
10     EMP00011   Nasrin Mondal  30.0    Male          A+              MTN   
68     EMP00069  Nadia Talukder  39.0    Male         AB+       Embroidery   
126    EMP00127  Sharmin Biswas  50.0  Female          B-          Cutting   
215    EMP00216   Kulsum Biswas  42.0  Female          O-            Admin   
228    EMP00229     Sohel Islam  26.0    Male          O-          Packing   
315    EMP00316  Mofiz Talukder  49.0    Male         AB-       embroidery   
323    EMP00324  Sharmin Mondal   NaN  Female          A-            Admin   
375    EMP00376   Kulsum Biswas  25.0  Female          O+          Packing   
406    EMP00407    Sohel Khatun  39.0    Male          O+            STORE   
484    

In [36]:
check_numeric_ranges(df_cleaned, 'age')
check_numeric_ranges(df_cleaned, 'basic_salary')
check_numeric_ranges(df_cleaned, 'bonus')
check_numeric_ranges(df_cleaned, 'attendance_pct')
check_numeric_ranges(df_cleaned, 'experience_yrs')



'age' — distribution:
  min:    0.0
  max:    999.0
  mean:   37.48
  median: 37.0
  nulls:  100

'basic_salary' — distribution:
  min:    0.0
  max:    54969.0
  mean:   19341.37
  median: 16088.0
  nulls:  15

'bonus' — distribution:
  min:    306
  max:    7953
  mean:   1950.62
  median: 1580.0
  nulls:  0

'attendance_pct' — distribution:
  min:    -10.0
  max:    999.0
  mean:   88.72
  median: 87.7
  nulls:  150

'experience_yrs' — distribution:
  min:    -5.0
  max:    99.0
  mean:   7.84
  median: 7.4
  nulls:  38


In [37]:
check_salary_discrepancy(df_cleaned, 'basic_salary', 'bonus', 'total_salary')

📊 --- Discrepancy Report ---
Equation Checked: basic_salary + bonus = total_salary
Total Rows with Math Errors: 447



,basic_salary,bonus,total_salary
9,32956.0,4497,37222.0
15,12372.0,894,13138.0
71,27828.0,3647,31271.0
73,16816.0,1829,NaN
84,12869.0,1847,14225.0
...,...,...,...
4910,17548.0,2034,19634.0
4916,24195.0,3083,27164.0
4923,8901.0,507,9773.0
4981,23286.0,1180,24366.0


In [38]:
# Calculate the exact difference amount
df_cleaned['salary_diff'] = (df_cleaned['basic_salary'].fillna(0) + df_cleaned['bonus'].fillna(0)) - df_cleaned['total_salary']

# Look at the unique differences
print(df_cleaned['salary_diff'].value_counts().head(10))

salary_diff
 0.0      4562
 357.0       2
 313.0       2
 270.0       2
 310.0       2
 169.0       2
-386.0       2
 77.0        2
 347.0       2
-255.0       2
Name: count, dtype: int64


In [39]:
import pandas as pd
import numpy as np

def validate_salary_totals(df, basic_col, bonus_col, total_col, flag_col='salary_status', tolerance=0.01):
    """
    Validates that basic_salary + bonus equals total_salary within a given tolerance.
    Adds a classification flag column without mutating the original input DataFrame.
    
    Parameters:
    -----------
    df : pd.DataFrame
        The target DataFrame.
    basic_col, bonus_col, total_col : str
        The names of the financial columns to cross-check.
    flag_col : str, default 'salary_status'
        The name of the new validation flag column to create.
    tolerance : float, default 0.01
        Allowed absolute difference to account for minor floating-point rounding errors.
    """
    # 1. Defensive Sandbox Copy
    df = df.copy()
    
    # 2. Schema Validation Guardrail
    for col in [basic_col, bonus_col, total_col]:
        if col not in df.columns:
            raise KeyError(f"Required column '{col}' missing from the DataFrame.")
            
    # 3. Safe, Vectorized Calculations (Handling NaNs as 0 for math safety)
    expected_total = df[basic_col].fillna(0) + df[bonus_col].fillna(0)
    actual_total = df[total_col].fillna(0)
    
    # 4. Calculate absolute difference to handle precision issues
    abs_diff = (expected_total - actual_total).abs()
    
    # 5. Vectorized Labeling
    df[flag_col] = np.where(abs_diff <= tolerance, 'Verified', 'Discrepancy')
    
    # 6. Pipeline Logging
    discrepancy_count = (df[flag_col] == 'Discrepancy').sum()
    print(f"✅ Validation Column '{flag_col}' created successfully.")
    print(f"📊 Audit Result: {discrepancy_count} discrepancies flagged out of {len(df)} total rows.\n")
    
    return df

In [40]:
df_cleaned = validate_salary_totals(df_cleaned, basic_col='basic_salary', bonus_col='bonus', total_col='total_salary')

✅ Validation Column 'salary_status' created successfully.
📊 Audit Result: 447 discrepancies flagged out of 5009 total rows.



In [41]:
df_cleaned.columns

Index(['employee_id', 'employee_name', 'age', 'gender', 'blood_group',
       'department', 'designation', 'division', 'basic_salary', 'bonus',
       'total_salary', 'phone', 'join_date', 'attendance_pct', 'email', 'nid',
       'experience_yrs', 'salary_diff', 'salary_status'],
      dtype='object')

In [ ]:
import pandas as pd
import numpy as np

def enforce_flexible_bounds(df, bounds_config):
    """
    Validates numeric columns against configurable minimum and/or maximum
    bounds. Values outside the specified limits are replaced with NaN.

    Args:
        df (pd.DataFrame): Input DataFrame.
        bounds_config (dict): Dictionary mapping column names to
            (min, max) tuples. Use None for one-sided bounds.
            Example: {"age": (18, 60), "salary": (0, None)}

    Returns:
        pd.DataFrame: A copy of the DataFrame with out-of-range values
        replaced by NaN.

    Example:
        >>> bounds = {
        ...     "age": (18, 60),
        ...     "salary": (0, None),
        ...     "attendance": (None, 100)
        ... }
        >>> df_clean = enforce_flexible_bounds(df, bounds)
    """

    df = df.copy()
        
    for col, bounds in bounds_config.items():
        if col not in df.columns:
            raise KeyError(f"Column '{col}' not found in DataFrame.")
            
        min_val, max_val = bounds
        
        out_of_bounds_mask = pd.Series(False, index=df.index)
        
        if min_val is not None:
            out_of_bounds_mask |= (df[col] < min_val)
            
        if max_val is not None:
            out_of_bounds_mask |= (df[col] > max_val)
            
        flagged_count = out_of_bounds_mask.sum()
        
        if flagged_count > 0:
            df.loc[out_of_bounds_mask, col] = np.nan
            
            range_desc = f"Min: {min_val if min_val is not None else 'None'} | Max: {max_val if max_val is not None else 'None'}"
            print(f"• Coerced {flagged_count:4d} rows to NaN in '{col}' ({range_desc})")
        else:
            print(f"• Verified '{col}': All values within specified limits.")
            
    print("-" * 50)
    return df

In [43]:
# Mapping your exact data rules
my_pipeline_rules = {
    'age': (18, 60),                  # Strict workforce bracket (drops 0 and 999)
    'basic_salary': (0, None),     # Must earn at least 10k, no upper cap ("to any")
    'bonus': (0, None),                # Bonus cannot be negative, no upper cap ("to any")
    'attendance_pct': (0, 100),        # Valid percentage boundary (drops -10 and 999)
    'experience_yrs': (0, 40)          # Career span tracking from entry to senior retirement
}



In [44]:
df_cleaned = enforce_flexible_bounds(df_cleaned, bounds_config=my_pipeline_rules)

🧹 --- Flexible Boundary Cleanup Report ---
• Coerced   30 rows to NaN in 'age' (Min: 18 | Max: 60)
• Verified 'basic_salary': All values within specified limits.
• Verified 'bonus': All values within specified limits.
• Coerced   40 rows to NaN in 'attendance_pct' (Min: 0 | Max: 100)
• Coerced   50 rows to NaN in 'experience_yrs' (Min: 0 | Max: 40)
--------------------------------------------------


In [45]:
check_numeric_ranges(df_cleaned, 'age')
check_numeric_ranges(df_cleaned, 'basic_salary')
check_numeric_ranges(df_cleaned, 'bonus')
check_numeric_ranges(df_cleaned, 'attendance_pct')
check_numeric_ranges(df_cleaned, 'experience_yrs')



'age' — distribution:
  min:    18.0
  max:    55.0
  mean:   36.63
  median: 37.0
  nulls:  130

'basic_salary' — distribution:
  min:    0.0
  max:    54969.0
  mean:   19341.37
  median: 16088.0
  nulls:  15

'bonus' — distribution:
  min:    306
  max:    7953
  mean:   1950.62
  median: 1580.0
  nulls:  0

'attendance_pct' — distribution:
  min:    75.0
  max:    100.0
  mean:   87.57
  median: 87.6
  nulls:  190

'experience_yrs' — distribution:
  min:    0.0
  max:    15.0
  mean:   7.47
  median: 7.4
  nulls:  88


In [ ]:

def impute_basic_salary_from_total(df, basic_col, bonus_col, total_col):
    """
    Deductively reconstructs missing or 0 basic salaries using the formula:
    Basic Salary = Total Salary - Bonus
    Only applies to rows where Basic Salary is 0 or NaN.
    """
    df = df.copy()
    
    for col in [basic_col, bonus_col, total_col]:
        if col not in df.columns:
            raise KeyError(f"Column '{col}' is missing from the DataFrame.")
            
    repair_mask = ((df[basic_col] == 0) | (df[basic_col].isna())) & (df[total_col] > df[bonus_col])
    
    rows_to_repair = repair_mask.sum()
    
    print("🛠️  --- Deductive Imputation Report ---")
    if rows_to_repair > 0:
        calculated_basic = df.loc[repair_mask, total_col] - df.loc[repair_mask, bonus_col]
        
        df.loc[repair_mask, basic_col] = calculated_basic
        print(f"• Successfully repaired {rows_to_repair} rows where basic salary was missing/0.")
        print(f"• Formula applied: {basic_col} = {total_col} - {bonus_col}")
    else:
        print("• No rows matched the criteria for basic salary imputation.")
        
    print("-" * 40)
    return df

In [47]:
df_cleaned = impute_basic_salary_from_total(df_cleaned, 'basic_salary', 'bonus', 'total_salary')

🛠️  --- Deductive Imputation Report ---
• Successfully repaired 60 rows where basic salary was missing/0.
• Formula applied: basic_salary = total_salary - bonus
----------------------------------------


In [48]:
df_cleaned = validate_salary_totals(
    df=df_cleaned, 
    basic_col='basic_salary', 
    bonus_col='bonus', 
    total_col='total_salary'
)

✅ Validation Column 'salary_status' created successfully.
📊 Audit Result: 387 discrepancies flagged out of 5009 total rows.



In [49]:
find_category_inconsistencies(df_cleaned, 'department')
find_category_inconsistencies(df_cleaned, 'designation')
find_category_inconsistencies(df_cleaned, 'division')



'department' — top 20 values (raw):
department
Cutting            387
Admin              381
Embroidery         375
Washing            355
Packing            350
Sewing             338
Maintenance        336
Store              335
Finishing          335
Quality Control    314
qc                  50
cutting             49
EMB                 47
pack                47
Maintnance          47
Cuttng              46
WASHING             45
admin               45
embroidery          45
Washng              44
Name: count, dtype: int64

'department' — top 20 values (normalized):
department
cutting            475
admin              465
embroidery         446
washing            438
packing            417
maintenance        414
store              410
sewing             406
finishing          395
quality control    368
qc                  73
pack                47
maintnance          47
emb                 47
cuttng              46
washng              44
embrodiery          43
pakcing             

In [ ]:


def clean_categorical_column(df, target_column, mapping_dict=None):
    """
    Cleans and standardizes a categorical column by removing leading and
    trailing whitespace, converting text to lowercase, optionally applying
    category mappings, and formatting the final values in title case.

    Args:
        df (pd.DataFrame):
            Input DataFrame.

        target_column (str):
            Name of the categorical column to clean.

        mapping_dict (dict, optional):
            Dictionary used to standardize category values after
            lowercase conversion. Keys should be lowercase to match
            the cleaned values.

    Returns:
        pd.DataFrame:
            A copy of the DataFrame with the cleaned categorical
            column. If the specified column does not exist, the
            DataFrame copy is returned unchanged.

    Example:
        >>> gender_map = {
        ...     "m": "male",
        ...     "male ": "male",
        ...     "f": "female",
        ...     "female ": "female"
        ... }
        >>>
        >>> df_clean = clean_categorical_column(
        ...     df,
        ...     target_column="gender",
        ...     mapping_dict=gender_map
        ... )
    """

    df = df.copy()

    if target_column not in df.columns:
        print(f"⚠️ Warning: '{target_column}' not found in DataFrame. Skipping.")
        return df

    df[target_column] = (
        df[target_column]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    if mapping_dict is not None:
        df[target_column] = df[target_column].replace(mapping_dict)

    df[target_column] = df[target_column].str.title()

    print(f"✅ Successfully cleaned column: '{target_column}'")

    return df

In [51]:
dept_map = {
    # Truncated names & abbreviations
    'cut': 'Cutting',
    'cuttng': 'Cutting',

    'mtn': 'Maintenance',
    'maintnance': 'Maintenance',
    'sew': 'Sewing',
    'sewing dept': 'Sewing',
    'sweing': 'Sewing',
    'finish': 'Finishing',
    'washng': 'Washing', 
    'emb': 'Embrodiery',
    'pack' : 'Packing',
    'pakcing' : 'Packing', 
      
    
    # Typos
    'strore': 'Store',
    'stores': 'Store',
    'finishin': 'Finishing',
    'quality ctrl': 'Quality Control',
    'qc': 'Quality Control',
    
    # Administrative alignment (Standardizing to 'Admin')
    'administration': 'Admin',
    'admn': 'Admin'
}

division_map = {
    'chittagong': 'Chattogram',
    'ctg': 'Chattogram',
    'khulana': 'Khulna',
    'ghazipur': 'Gazipur',
    'rajshai': 'Rajshahi',
    'n.ganj': 'Narayanganj',
    'naraynganj': 'Narayanganj',
    'syhlet': 'Sylhet',
    'dkha': 'Dhaka'
}

In [52]:
df_cleaned=clean_categorical_column(df_cleaned, target_column='department', mapping_dict=dept_map)



✅ Successfully cleaned column: 'department'


In [53]:
df_cleaned=clean_categorical_column(df_cleaned, target_column='division', mapping_dict=division_map)


✅ Successfully cleaned column: 'division'


In [54]:
check_global_structure(df)


=== Global Dataset Structure Audit ===
Total Rows:    5120
Total Columns: 17
Completely Duplicated Rows: 111 (2.17%)

Checking for constant (single-value) columns...
  No constant columns found. All features contain variation.


In [55]:
check_global_structure(df_cleaned)


=== Global Dataset Structure Audit ===
Total Rows:    5009
Total Columns: 19
Completely Duplicated Rows: 8 (0.16%)

Checking for constant (single-value) columns...
  No constant columns found. All features contain variation.


In [56]:
for col in df_cleaned.columns:

    column_based_inspection(df_cleaned, col)


📊 --- Inspection Report: employee_id ---
Total Rows:     5009
Data Type:      object
Null Values:    0
Duplicate Rows: 9


📊 --- Inspection Report: employee_name ---
Total Rows:     5009
Data Type:      object
Null Values:    0
Duplicate Rows: 3925


📊 --- Inspection Report: age ---
Total Rows:     5009
Data Type:      float64
Null Values:    130
Duplicate Rows: 4970


📊 --- Inspection Report: gender ---
Total Rows:     5009
Data Type:      object
Null Values:    36
Duplicate Rows: 5006


📊 --- Inspection Report: blood_group ---
Total Rows:     5009
Data Type:      object
Null Values:    71
Duplicate Rows: 5000


📊 --- Inspection Report: department ---
Total Rows:     5009
Data Type:      object
Null Values:    0
Duplicate Rows: 4998


📊 --- Inspection Report: designation ---
Total Rows:     5009
Data Type:      object
Null Values:    75
Duplicate Rows: 4998


📊 --- Inspection Report: division ---
Total Rows:     5009
Data Type:      object
Null Values:    0
Duplicate Rows: 5002


📊 -

In [57]:
df_cleaned = clean_categorical_column(df_cleaned, 'employee_name')


✅ Successfully cleaned column: 'employee_name'


In [58]:
find_mixed_script_names(df_cleaned, 'employee_name')

🔍 Found 151 names with mixed Bengali and English scripts.


,employee_id,employee_name,age,gender,blood_group,department,designation,division,basic_salary,bonus,total_salary,phone,join_date,attendance_pct,email,nid,experience_yrs,salary_diff,salary_status
53,EMP00054,করিম Khan,28.0,Male,AB-,Cutting,Operator,Dhaka,9407.0,727,10134.0,01824883373,2022-03-27,82.1,anwar.khatun87@gmail.com,9562417313120,7.9,0.0,Verified
121,EMP00122,করিম Uddin,26.0,Female,A+,Quality Control,Senior Operator,Chattogram,16145.0,943,17088.0,01950486688,2021-02-07,NaN,rashid.hawladar78@gmail.com,4317903135288,11.2,0.0,Verified
163,EMP00164,করিম Khan,28.0,Male,A-,Store,Floor In-charge,Narayanganj,33305.0,3808,37113.0,01869618021,2020-04-05,95.0,roksana.uddin58@gmail.com,8866225971831,4.7,0.0,Verified
203,EMP00204,রোকসানা Talukder,35.0,Female,B+,Store,Machine Mechanic,Rajshahi,14626.0,2153,17272.0,01968843284,2019-01-08,93.2,sumaiya.sarker30@gmail.com,3650382214740,0.2,-493.0,Discrepancy
357,EMP00358,হাসিনা Sarker,47.0,Male,A-,Finishing,Line Chief,Gazipur,27171.0,3645,30816.0,01749479427,2018-12-16,78.1,mofiz.begum95@gmail.com,3960911152782,1.2,0.0,Verified
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4821,EMP04822,করিম Uddin,19.0,Female,B-,Packing,Operator,Chattogram,9130.0,1052,10182.0,01786163538,NaT,86.7,belal.ali66@gmail.com,9095268965707,8.3,0.0,Verified
4830,EMP04831,কুলসুম Ali,NaN,Male,AB-,Washing,Trainee,Narayanganj,6139.0,326,6465.0,01954860655,NaT,77.6,nasrin.talukder92@gmail.com,8216532394321,NaN,0.0,Verified
4918,EMP04919,করিম Mondal,47.0,Male,A-,Embrodiery,Operator,Narayanganj,12411.0,1591,14002.0,01824346049,2022-08-04,78.7,hasina.khatun40@gmail.com,3652580238024,5.3,0.0,Verified
4938,EMP04939,মফিজ Ali,34.0,Female,B+,Quality Control,Senior Operator,Narayanganj,15333.0,1262,16595.0,01835683424,2022-03-04,75.8,rubel.khatun47@gmail.com,1525149340210,15.0,0.0,Verified


In [ ]:
def audit_employee_names(df, name_column='employee_name'):
    """
    Audits name columns for mixed-script anomalies. 
    Flags rows containing both Bengali and English text.
    """
    df = df.copy()
    
    bengali_pattern = r'[\u0980-\u09FF]'
    english_pattern = r'[a-zA-Z]'
    
    has_bengali = df[name_column].astype(str).str.contains(bengali_pattern, regex=True)
    has_english = df[name_column].astype(str).str.contains(english_pattern, regex=True)
    
    df['name_status'] = 'Verified'
    df.loc[has_bengali & has_english, 'name_status'] = 'Mixed Script Discrepancy'
    
    anomaly_count = (df['name_status'] == 'Mixed Script Discrepancy').sum()
    print(f"📋 Name Audit Complete: Found {anomaly_count} mixed-script discrepancies.")
    
    return df

In [60]:
df_cleaned = audit_employee_names(df_cleaned, name_column='employee_name')

📋 Name Audit Complete: Found 151 mixed-script discrepancies.


In [61]:
Path('clean').mkdir(parents=True, exist_ok=True)



In [62]:
df_cleaned.to_excel('clean/rmg_employee_clean.xlsx', index = False)
# print(f"\n{file} -> {output_path}")